# Data Preprocessing

## Objective

The purpose of this notebook is to clean and prepare the Ames Housing dataset for machine learning. Based on the findings from the exploratory data analysis (EDA), this notebook handles missing values, performs feature engineering, encodes categorical variables, treats outliers where appropriate, and prepares the final dataset for model training.

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("../data/AmesHousing.csv")

df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (2930, 82)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   object 
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2930 non-null   object 
 9   Land Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot Config       2930 non-null   object 
 12  Land Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition 1      2930 non-null   object 
 15  Condition 2      2930 non-null   object 
 16  Bldg Type        2930 non-null   o

In [4]:
columns_to_drop = ["Order", "PID"]

df = df.drop(columns=columns_to_drop)

# Handle Missing Values

## Objective

Based on the EDA findings, missing values in the Ames Housing dataset are not entirely random. Many missing values indicate the absence of a property feature (e.g., no pool or no alley), while others represent genuinely missing information. This section applies appropriate imputation strategies to prepare the dataset for machine learning.

In [5]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

missing

Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Qual        159
Garage Cond        159
Garage Yr Blt      159
Garage Finish      159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Cond           80
Bsmt Qual           80
BsmtFin Type 1      80
Mas Vnr Area        23
Bsmt Full Bath       2
Bsmt Half Bath       2
BsmtFin SF 1         1
BsmtFin SF 2         1
Electrical           1
Total Bsmt SF        1
Bsmt Unf SF          1
Garage Area          1
Garage Cars          1
dtype: int64

In [6]:
missing_df = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum()/len(df))*100
})

missing_df = missing_df[missing_df["Missing Count"] > 0]

missing_df.sort_values(
    by="Missing Percentage",
    ascending=False
)

,Missing Count,Missing Percentage
Pool QC,2917,99.556314
Misc Feature,2824,96.382253
Alley,2732,93.242321
Fence,2358,80.477816
Mas Vnr Type,1775,60.580205
Fireplace Qu,1422,48.532423
Lot Frontage,490,16.723549
Garage Qual,159,5.426621
Garage Cond,159,5.426621
Garage Yr Blt,159,5.426621


In [7]:
none_features = [
    "Pool QC",
    "Misc Feature",
    "Alley",
    "Fence",
    "Fireplace Qu",
    "Garage Finish",
    "Garage Qual",
    "Garage Cond",
    "Garage Type",
    "Bsmt Exposure",
    "BsmtFin Type 1",
    "BsmtFin Type 2",
    "Bsmt Qual",
    "Bsmt Cond",
    "Mas Vnr Type"
]

for col in none_features:
    df[col] = df[col].fillna("None")

In [8]:
median_features = [
    "Lot Frontage",
    "Mas Vnr Area",
    "Garage Yr Blt"
]

for col in median_features:
    df[col] = df[col].fillna(df[col].median())

In [9]:
mode_features = [
    "Electrical"
]

for col in mode_features:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
print("Remaining Missing Values:")

df.isnull().sum().sum()

Remaining Missing Values:


np.int64(10)

In [11]:
remaining_missing = df.isnull().sum()

remaining_missing = remaining_missing[remaining_missing > 0]

remaining_missing.sort_values(ascending=False)

Bsmt Half Bath    2
Bsmt Full Bath    2
BsmtFin SF 1      1
BsmtFin SF 2      1
Total Bsmt SF     1
Bsmt Unf SF       1
Garage Cars       1
Garage Area       1
dtype: int64

In [12]:
bsmt_num_features = [
    "BsmtFin SF 1",
    "BsmtFin SF 2",
    "Bsmt Unf SF",
    "Total Bsmt SF",
    "Bsmt Full Bath",
    "Bsmt Half Bath"
]

for col in bsmt_num_features:
    df[col] = df[col].fillna(0)

In [13]:
garage_num_features = [
    "Garage Cars",
    "Garage Area"
]

for col in garage_num_features:
    df[col] = df[col].fillna(0)

In [14]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

Remaining Missing Values:
0


# Missing Value Handling – Summary

## Objective

The missing values in the Ames Housing dataset were handled using feature-specific strategies based on the findings from Exploratory Data Analysis (EDA).

## Strategies Applied

### 1. Features Representing Absence

Missing values in features such as **Pool QC, Alley, Fence, Fireplace Qu, Garage Type, Garage Finish, Garage Qual, Garage Cond, Basement Quality, Basement Exposure**, and related attributes were replaced with **"None"**, indicating that the corresponding feature is not present in the property.

### 2. Numerical Features

Numerical variables such as **Lot Frontage, Mas Vnr Area, and Garage Yr Blt** were imputed using the **median**, as it is less affected by outliers than the mean.

### 3. Basement Numerical Features

Missing values in **BsmtFin SF 1, BsmtFin SF 2, Bsmt Unf SF, Total Bsmt SF, Bsmt Full Bath, and Bsmt Half Bath** were replaced with **0**, representing houses without a basement.

### 4. Garage Numerical Features

Missing values in **Garage Cars** and **Garage Area** were replaced with **0**, representing houses without a garage.

### 5. Remaining Categorical Features

The **Electrical** feature contained only a few missing values and was imputed using the most frequent category (mode).

---

## Outcome

- All missing values have been successfully handled.
- The dataset now contains **zero missing values**.
- The preprocessing preserves the real-world meaning of the data while preparing it for machine learning.

# Recalculate Engineered Features

Since some missing values were filled after the initial feature engineering, the engineered features are recalculated to ensure they contain no missing values.

In [15]:
# Recalculate Total Bathrooms

df["Total Bathrooms"] = (
    df["Full Bath"]
    + 0.5 * df["Half Bath"]
    + df["Bsmt Full Bath"]
    + 0.5 * df["Bsmt Half Bath"]
)

In [16]:
# Recalculate Total Living Area

df["Total Living Area"] = (
    df["Gr Liv Area"]
    + df["Total Bsmt SF"]
)

In [17]:
print("Remaining Missing Values:", df.isnull().sum().sum())

missing = df.isnull().sum()

print("\nColumns with Missing Values:")
print(missing[missing > 0])

Remaining Missing Values: 0

Columns with Missing Values:
Series([], dtype: int64)


In [18]:
print("Total Missing Values:", df.isnull().sum().sum())

Total Missing Values: 0


In [19]:
missing = df.isnull().sum()

print(missing[missing > 0])

Series([], dtype: int64)


In [20]:
cols = [
    "Lot Frontage",
    "Mas Vnr Area",
    "BsmtFin SF 1",
    "BsmtFin SF 2",
    "Bsmt Unf SF",
    "Total Bsmt SF",
    "Bsmt Full Bath",
    "Bsmt Half Bath",
    "Garage Yr Blt",
    "Garage Cars",
    "Garage Area",
    "Total Bathrooms",
    "Total Living Area"
]

print(df[cols].isnull().sum())

Lot Frontage         0
Mas Vnr Area         0
BsmtFin SF 1         0
BsmtFin SF 2         0
Bsmt Unf SF          0
Total Bsmt SF        0
Bsmt Full Bath       0
Bsmt Half Bath       0
Garage Yr Blt        0
Garage Cars          0
Garage Area          0
Total Bathrooms      0
Total Living Area    0
dtype: int64


In [21]:
print(df.isnull().sum().sum())

0


In [22]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)


In [23]:
df

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Total Bathrooms,Total Living Area
0,20,RL,141.0,31770,Pave,None,IR1,Lvl,AllPub,Corner,...,None,None,0,5,2010,WD,Normal,215000,2.0,2736.0
1,20,RH,80.0,11622,Pave,None,Reg,Lvl,AllPub,Inside,...,MnPrv,None,0,6,2010,WD,Normal,105000,1.0,1778.0
2,20,RL,81.0,14267,Pave,None,IR1,Lvl,AllPub,Corner,...,None,Gar2,12500,6,2010,WD,Normal,172000,1.5,2658.0
3,20,RL,93.0,11160,Pave,None,Reg,Lvl,AllPub,Corner,...,None,None,0,4,2010,WD,Normal,244000,3.5,4220.0
4,60,RL,74.0,13830,Pave,None,IR1,Lvl,AllPub,Inside,...,MnPrv,None,0,3,2010,WD,Normal,189900,2.5,2557.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,80,RL,37.0,7937,Pave,None,IR1,Lvl,AllPub,CulDSac,...,GdPrv,None,0,3,2006,WD,Normal,142500,2.0,2006.0
2926,20,RL,68.0,8885,Pave,None,IR1,Low,AllPub,Inside,...,MnPrv,None,0,6,2006,WD,Normal,131000,2.0,1766.0
2927,85,RL,62.0,10441,Pave,None,Reg,Lvl,AllPub,Inside,...,MnPrv,Shed,700,7,2006,WD,Normal,132000,1.5,1882.0
2928,20,RL,77.0,10010,Pave,None,Reg,Lvl,AllPub,Inside,...,None,None,0,4,2006,WD,Normal,170000,2.0,2778.0


In [24]:
df.to_csv("../data/cleaned_housing_data.csv", index=False)